<a href="https://colab.research.google.com/github/NoaTal1996/Backbone-Graph/blob/main/src/key_authors_graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

Note: google drive is no longer suppurted. Adjset the input and output path if nedded.

In [ ]:
# General
from pathlib import Path

# ML
import pandas as pd
import matplotlib.pyplot as plt

# TF-IDf
from sklearn.feature_extraction.text import TfidfVectorizer

# Graphs
import networkx as nx
from sklearn.feature_selection import mutual_info_classif

# Loading and Exporting

## Parameters

In [ ]:
dataset_name = "Labeled_Datasets/Ecuador"   # folder path
file_name = "Ecuador_part_5.gzip.parquet"   # file name
dataset_procced_date = "2025_12_01"         # ("%Y_%m_%d")

# Recommended: working directory is: "Backbone-Graph/src"

## Load Data

In [ ]:
from pathlib import Path

# Google Colab
if in_google_colab:
    raise Exception("Google Colab not supported yet.")
    print("Running inside Colab. Loading from Google Drive...")

# Cluster\Local
else:
    # Recommended: working directory is: "Backbone-Graph/src"
    # Loading result of "Topic_Clustring_and_NER.ipynb".
    file_name_no_suffixes = file_name.split(".")[0]
    processed_dataset_path = Path("../results") / dataset_name / file_name_no_suffixes / dataset_procced_date
    # Example: ""../results/Labeled_Datasets/Ecuador/Ecuador_part_4/2025_11_27"
    print(f"Running on local/cluster.\nLoading data from: {processed_dataset_path}")


# Load the parquet file
df = pd.read_parquet(processed_dataset_path)

# for testing
df = df.sample(n = 50).reset_index(drop=True)

## Output

In [ ]:
output_path = processed_dataset_path # save in the save folder as the processed dataset

# Dataset collums

In [ ]:
print(f"{'**collum name**':<30} **value type**")
for col in df.columns:
    # dtype_str = str(df[col].dtype)  # Convert dtype to string
    # get a sample non-null value if exists
    sample_val = df[col].dropna().iloc[0] if df[col].notna().any() else None
    print(f"{col:<30} | {type(sample_val).__name__}")

# TF-IDF Score

## Entities

TF-IDF Using *sklearn*

In [ ]:

def compute_entities_tfidf(df, entity_col, topic_col="BERTopic_topic"):
    """
    Compute TF-IDF over entities grouped by topic.
    Output shape: rows = entities, columns = topics.

    Args:
        df (pd.DataFrame): Contains [topic_col] and [entity_col] (list of entities).
        entity_col (str): Column with lists of entities per post.
        topic_col (str): Topic label column.

    Returns:
        pd.DataFrame: TF-IDF matrix (rows = entities, columns = topics).
    """

    # 1) Explode rows → (topic, entity)
    df_exp = (
        df[[topic_col, entity_col]]
        .explode(entity_col)
        .dropna(subset=[entity_col])
        .astype({entity_col: str})
    )

    # 2) Build topic "documents"
    topic_docs_df = (
        df_exp.groupby(topic_col)[entity_col]
              .apply(lambda x: " ".join(x))
              .reset_index()
              .rename(columns={entity_col: "doc"})
    )

    # 3) TF-IDF where rows=topics, cols=entities
    vec = TfidfVectorizer(token_pattern=r"\S+")
    X = vec.fit_transform(topic_docs_df["doc"])

    topic_entity_df = pd.DataFrame(
        X.toarray(),
        index=topic_docs_df[topic_col].values,
        columns=vec.get_feature_names_out()
    )

    # 4) Transpose → rows=entities, columns=topics
    entity_topic_df = topic_entity_df.T

    return entity_topic_df

In [ ]:
hashtags_tfidf_df = compute_entities_tfidf(df, entity_col='hashtags', topic_col="BERTopic_topic")
hashtags_tfidf_df.head()

In [ ]:
def get_top_entities_list(entity_topic_df, topic_col, top_n=10):
    """
    Return a list of the top-N entities for a specific topic.

    Args:
        entity_topic_df (pd.DataFrame):
            Rows = entities, columns = topics.
        topic_col (str | int):
            Topic column name.
        top_n (int):
            Number of top entities.

    Returns:
        list[str]: Top-N entities sorted by descending TF-IDF.
    """

    top_entities = entity_topic_df[topic].sort_values(ascending=False).head(top_n).index.tolist()

    return top_entities

In [ ]:
top_entities = get_top_entities(tfidf_df, topic_id=1, topic_col = "BERTopic_topic", n=10)
print(top_entities)

## Posts

TF-IDF<sub>topic,post</sub> = ∑<sub>entity ∈ post</sub> TF-IDF<sub>topic,entity</sub>

In [ ]:
# prams
topic_col = "BERTopic_topic"
entity_cols_lst = ["hashtags", "account_mentions", "urls", "ner_entities"]

In [ ]:
def compute_post_tfidf(
    df: pd.DataFrame,
    topic_col: str | int,
    entity_cols_lst: list[str],
    out_col: str = "post_tfidf",
):
    """Compute per-post TF-IDF scores from multiple entity columns.

    For each entity column, computes an entity×topic TF-IDF matrix using
    `compute_entities_tfidf` and adds the sum of TF-IDF(entity, topic)
    for all entities in each post to `out_col`.

    Args:
        df: Input DataFrame.
        entity_cols_lst: Columns with lists of entities.
        topic_col: Column containing topic labels.
        out_col: Name of the output score column.
    """
    df[out_col] = 0.0

    for entity_col in entity_cols_lst:
        entities_tfidf_df = compute_entities_tfidf(df, entity_col=entity_col)

        df[out_col] += df.apply(
            lambda row: entities_tfidf_df.loc[row[entity_col], row[topic_col]].sum(),
            axis=1,
        )

In [ ]:
compute_post_tfidf(df, topic_col, entity_cols_lst)
df[["post_text", "post_tfidf"]].head()

## Authors

*(a = author, t= topic, p = post)*

TF<sub>a, t</sub> = ∑ <sub> p ∈ authors's posts in topic</sub> tfidf_score<sub>p, t</sub>

IDF<sub>a</sub> = log((|all_topics| + offset) / (|author's_topics| + offset))

TF-IDF<sub>a,t</sub> = TF<sub>a, t</sub> * IDF<sub>a</sub>

In [ ]:
def compute_author_tfidf_to_df(
    df: pd.DataFrame,
    topic_col: str = "BERTopic_topic",
    author_col: str = "author",
    post_score_col: str = "post_tfidf",
    out_col: str = "topic_author_tfidf",
    use_log_idf: bool = True,
    idf_offset: float = 1.0,
) -> pd.DataFrame:
    """Compute author–topic TF-IDF and write each post's score to df.

    TF = sum of post scores per (author, topic).
    IDF = penalizes authors active in many topics.
    Output column contains TF-IDF(author, topic) for each post.

    Args:
        df: Input DataFrame with posts.
        topic_col: Topic column name.
        author_col: Author column name.
        post_score_col: Per-post score column.
        out_col: Output column added to df.
        use_log_idf: Apply log to IDF.
        idf_offset: Smoothing constant.

    Returns:
        author_topic_tfidf: Author × topic TF-IDF matrix.
    """

    # TF matrix: rows = authors, columns = topics, values = tf score
    author_topic_tf = (
        df.groupby([author_col, topic_col])[post_score_col]
        .sum()
        .unstack(fill_value=0.0)
    )

    # IDF per author
    n_topics = author_topic_tf.shape[1]
    topics_per_author = (author_topic_tf > 0).sum(axis=1)
    idf = (n_topics + idf_offset) / (topics_per_author + idf_offset)
    if use_log_idf:
        idf = np.log(idf)

    # TF-IDF matrix
    author_topic_tfidf = author_topic_tf.mul(idf, axis=0)

    # Assign TF-IDF(author, topic) to each post
    df[out_col] = df.apply(
        # Pandas aligns by row index = author.
        lambda row: author_topic_tfidf.loc[row[author_col], row[topic_col]],
        axis=1,
    )

    return author_topic_tfidf

In [ ]:
author_topic_tfidf = compute_author_tfidf_to_df(df)
author_topic_tfidf.head()

# Key Authors

In [ ]:
def get_top_authors_from_df(
    df: pd.DataFrame,
    topic: str | int,
    score_col: str = "topic_author_tfidf",
    author_col: str = "author",
    topic_col: str = "BERTopic_topic",
    n: int = 10
) -> set:
    """Return the top-n authors for a given topic based on TF-IDF author scores.

    Args:
        df: DataFrame containing posts with author-topic TF-IDF scores.
        topic: Topic to extract top authors for.
        score_col: Column storing TF-IDF(author, topic) scores.
        author_col: Column storing author IDs.
        topic_col: Column storing topic labels.
        n: Number of top authors to return.

    Returns:
        set: Top-n authors with the highest TF-IDF scores for the topic.
    """
    # Filter to rows matching the topic and rank authors by summed scores
    top_authors = (
        df.loc[df[topic_col] == topic, [author_col, score_col]]
        .groupby(author_col)[score_col]
        .sum()
        .sort_values(ascending=False)
        .head(n)
        .index
    )

    return set(top_authors)

In [ ]:
print(get_top_authors_from_df(df, topic=1))

# Networkx Graphs

## Build Bipartite Account ↔ Account Graph

In [ ]:
def add_relation_edges(
    G: nx.Graph,
    df: pd.DataFrame,
    src_col: str,
    dst_col: str,
    weight: int = 1,
):
    """Add edges to a graph from df[src_col] → df[dst_col].

    Automatically explodes src or dst if they contain list-like values.

    Args:
        G (nx.Graph): Graph to update (modified in-place).
        df (pd.DataFrame): Input DataFrame.
        src_col (str): Column containing sources (single or list-like).
        dst_col (str): Column containing targets (single or list-like).
        weight (int): Weight added per relation (default = 1).

    Returns:
        None: Graph is modified directly.
    """

    # keep valid rows
    pairs = df[[src_col, dst_col]].dropna()

    # auto-detect list-like columns
    is_src_list = pairs[src_col].map(lambda x: isinstance(x, (list, tuple, set))).any()
    is_dst_list = pairs[dst_col].map(lambda x: isinstance(x, (list, tuple, set))).any()

    # explode step-by-step
    if is_src_list:
        pairs = pairs.explode(src_col).dropna(subset=[src_col])
    if is_dst_list:
        pairs = pairs.explode(dst_col).dropna(subset=[dst_col])

    # aggregate (src, dst) counts
    counts = (
        pairs
        .value_counts([src_col, dst_col])
        .reset_index(name="count")
    )

    # build weighted edges
    for _, row in counts.iterrows():
        src = row[src_col]
        dst = row[dst_col]
        w = row["count"] * weight

        if src not in G:
            G.add_node(src)
        if dst not in G:
            G.add_node(dst)

        if G.has_edge(src, dst):
            G[src][dst]["weight"] += w
        else:
            G.add_edge(src, dst, weight=w)

In [ ]:
account_relations = [
    ("accountid", "in_reply_to_accountid"),   # reply
    ("accountid", "account_mentions"),        # mentions (list)
    ("accountid", "reposted_accountid"),      # repost
]

G_accounts = nx.DiGraph()

for src_col, dst_col in account_relations:

    add_relation_edges(
        G_accounts,
        df,
        src_col=src_col,
        dst_col=dst_col,
        weight=1,
    )

## Plot Graph

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from pathlib import Path

def plot_and_save_graph(
    G: nx.Graph,
    output_path: str = "",
    graph_name: str = "graph",
    figsize=(12, 12),
):
    """Plot a NetworkX graph, display it, and save it if provided with output_path as a PNG file.

    Args:
        G (nx.Graph): Graph to visualize.
        output_path (str): Directory where the image will be saved (default: "").
        graph_name (str): Base name of the saved file (default: 'graph').
        figsize (tuple): Figure size in inches.

    Returns:
        None
    """
    plt.figure(figsize=figsize)

    # node positions (stable layout)
    pos = nx.spring_layout(G, seed=42)

    # draw graph
    nx.draw(
        G,
        pos,
        node_size=30,
        node_color="steelblue",
        edge_color="gray",
        width=0.5,
        alpha=0.8,
        with_labels=False,
    )

    plt.tight_layout()

    # build output path
    if output_path is not None:
      plot_path = Path(save_dir) / f"{graph_name}.png"
      plt.savefig(plot_path, dpi=300)
      print(f"Saved graph to: {plot_path}")

    # Show figure
    plt.show()

    # Close figure
    plt.close()

# Centralities

## Compute centralities

In [ ]:
def compute_all_centralities(G: nx.Graph) -> pd.DataFrame:
    """Compute multiple graph centralities.

    Returns a DataFrame indexed by author/node with:
      - degree
      - betweenness
      - closeness
      - eigenvector
      - pagerank
      - kcore (core number)

    Args:
        G: NetworkX graph

    Returns:
        centralities_df: Centrality measures per node.
    """
    # Degree centrality
    degree = nx.degree_centrality(G)

    # Betweenness centrality
    betweenness = nx.betweenness_centrality(G, normalized=True)

    # Closeness centrality
    closeness = nx.closeness_centrality(G)

    # Eigenvector centrality
    eigen = nx.eigenvector_centrality(G)

    # PageRank
    pagerank = nx.pagerank(G)

    # K-core number
    kcore = nx.core_number(G)

    # Build DataFrame
    node_centralities_df = pd.DataFrame({
        "degree_centrality": degree,
        "betweenness_centrality": betweenness,
        "closeness_centrality": closeness,
        "eigenvector_centrality": eigen,
        "pagerank": pagerank,
        "kcore": kcore,
    })

    return node_centralities_df

In [ ]:
node_centralities_df = compute_all_centralities(G_accounts)
node_centralities_df.head()

## Information Gain

In [ ]:
def compute_centrality_information_gain(
    df: pd.DataFrame,
    node_centralities_df: pd.DataFrame,
    io_tag_col: str = "IO_Driver",
    author_col: str = "author",
) -> pd.Series:
    """Compute information gain (mutual information) between centrality features
    and an IO-driver tag.

    Args:
        df: DataFrame with one row per post. Must contain the IO tag column
            and an author_col column.
        io_tag_col: Column in df containing the IO-driver tag.
        author_col: Column identifying the author of each post.
        node_centralities_df: DataFrame with authors as index and centrality metrics as columns.

    Returns:
        pd.Series: Information gain per centrality feature, sorted descending.
    """
    # Aggregate IO-driver label to author-level
    author_io_tag = df.groupby(author_col)[io_tag_col].mean().round().astype(int)

    # Align author indices between centralities and labels
    centralities_df, author_io_tag = node_centralities_df.align(
        author_io_tag, join="inner", axis=0
    )

    # Prepare arrays for sklearn
    X = centralities_df.values
    y = author_io_tag.values

    # Compute information gain
    mi = mutual_info_classif(X, y, discrete_features=False, random_state=42)

    # Convert to Series, sort, and return
    mi_series = pd.Series(
        mi, index=centralities_df.columns, name="information_gain"
    ).sort_values(ascending=False)

    return mi_series

In [ ]:
mi_series = compute_centrality_information_gain(df, node_centralities_df)

In [ ]:
def plot_centrality_information_gain(
    mi_series: pd.Series,
    output_path: Path | str = None,
    filename: str = "centrality_information_gain.png",
) -> None:
    """Plot and save information gain per centrality feature.

    Args:
        mi_series: Information gain per centrality feature.
        output_path: Directory where the image will be saved.
        filename: Name of the output image file.
    """

    # Create the plot
    plt.figure(figsize=(8, 4))
    mi_series.plot(kind="bar")

    plt.ylabel("Information gain (mutual information)")
    plt.xlabel("Centrality feature")
    plt.title("Information gain of centralities for IO_driver")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    # Save figure
    if output_path is not None:
      output_path = Path(output_path)
      file_path = output_path / filename
      plt.savefig(file_path, dpi=300)
      print(f"Saved fig to: {file_path}")

    # Show figure
    plt.show()

    # Close figure
    plt.close()

In [ ]:
plot_centrality_information_gain(mi_series)